# Lab 09-02: Trace, Log, and Alert on Tool Calls

**Book chapter:** Chapter 09: Deploying, Operating, and Observing Agents  
**Estimated time:** 25 minutes

## Overview

Shoppers report that Cymbal Retail's stock assistant is sometimes slow and sometimes
cannot answer. In this lab, you send the agent's traces to Cloud Trace, write a
structured log for every tool call that links to its trace, create a log-based metric
and an alert policy for failed tool calls, and use the trace waterfall and the logs to
find the cause.

## Learning Objectives

By the end of this lab, you will be able to:

- Export ADK's OpenTelemetry traces to Cloud Trace
- Write structured JSON logs linked to traces from a callback
- Create a log-based metric and an alert policy in code
- Find a latency bottleneck in a trace waterfall
- Query structured logs by field

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** Less than USD 1. Run the **Clean Up** cell at the end.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Signal | Answers | Where |
|---|---|---|
| **Trace** | Where did the time go, and what did the agent do? | Cloud Trace |
| **Structured log** | Which requests failed, and with what details? | Cloud Logging |
| **Metric** | How often is it happening, and is it getting worse? | Cloud Monitoring |
| **Alert** | Does someone need to act now? | Cloud Monitoring |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk google-cloud-logging google-cloud-monitoring

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Send the agent's traces to Cloud Trace and to local memory

ADK is instrumented with **OpenTelemetry**. `get_gcp_exporters` exports spans to Cloud
Trace, and an in-memory exporter keeps a copy so that you can print the waterfall in
Step 5. Setting `ADK_CAPTURE_MESSAGE_CONTENT_IN_SPANS` to `false` keeps prompts and
responses out of the spans, which protects personal data.

> **Note:** Exporting spans uses the Telemetry API (`telemetry.googleapis.com`), which the
> Appendix C setup enables.

In [ ]:
import os
import time
import uuid

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
# Keep prompts and responses out of the spans, as in production.
os.environ["ADK_CAPTURE_MESSAGE_CONTENT_IN_SPANS"] = "false"

from google.adk.telemetry.google_cloud import get_gcp_exporters
from google.adk.telemetry.setup import OTelHooks, maybe_set_otel_providers
from opentelemetry import trace
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter)

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
spans = InMemorySpanExporter()  # A local copy of the spans to inspect.
provider = trace.get_tracer_provider()
if hasattr(provider, "add_span_processor"):  # Tracing is already set up.
    provider.add_span_processor(SimpleSpanProcessor(spans))
else:
    maybe_set_otel_providers([
        get_gcp_exporters(enable_cloud_tracing=True),
        OTelHooks(span_processors=[SimpleSpanProcessor(spans)])])
print("Tracing to Cloud Trace in project", PROJECT)

## Step 2: Build an agent that writes a structured log for each tool call

`check_inventory` simulates a warehouse API that times out after 2 seconds for camping
tents. Two callbacks time each tool call, and `log_tool_result` writes a **structured log**
entry: a JSON payload with fields that can be filtered and counted, plus the trace ID and
span ID of the current span, so each entry links to its trace.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.cloud import logging as cloud_logging
from google.cloud.logging import Resource
from google.genai import types

log_client = cloud_logging.Client(project=PROJECT)
logger = log_client.logger("cymbal-agent")
STOCK = {"trailhead boots": 12, "rain jacket": 0}
STARTED = {}


def check_inventory(item: str) -> dict:
    """Returns the warehouse stock level for an item."""
    if "tent" in item.lower():  # Simulate a failing warehouse system.
        time.sleep(2)
        return {"status": "error", "error": "Warehouse API timed out"}
    time.sleep(0.2)
    return {"status": "ok", "units": STOCK.get(item.lower(), 0)}


def start_timer(tool, args, tool_context):
    STARTED[tool_context.function_call_id] = time.perf_counter()


def log_tool_result(tool, args, tool_context, tool_response):
    """Writes one JSON log entry, linked to the current trace."""
    started = STARTED.pop(tool_context.function_call_id)
    status = tool_response.get("status", "ok")
    span = trace.get_current_span().get_span_context()
    logger.log_struct(
        {"event": "tool_result", "agent": tool_context.agent_name,
         "tool": tool.name, "status": status,
         "latency_ms": round((time.perf_counter() - started) * 1000),
         "session_id": tool_context.session.id,
         "invocation_id": tool_context.invocation_id},
        severity="ERROR" if status == "error" else "INFO",
        resource=Resource(type="global", labels={"project_id": PROJECT}),
        trace=f"projects/{PROJECT}/traces/{span.trace_id:032x}",
        span_id=f"{span.span_id:016x}")


agent = Agent(
    name="stock_assistant", model=MODEL,
    instruction="You answer stock questions for Cymbal Retail. Call "
                "check_inventory once per item. If it returns an error, "
                "apologize and suggest trying again later. Be brief.",
    tools=[check_inventory], before_tool_callback=start_timer,
    after_tool_callback=log_tool_result)
runner = Runner(app_name="stock_app", agent=agent,
                session_service=InMemorySessionService(),
                auto_create_session=True)

## Step 3: Create a log-based metric and an alert policy

A **log-based metric** counts log entries that match a filter, here the failed tool calls.
The **alert policy** opens an incident when the metric exceeds 2 in a 5-minute window. A
log-based metric counts only entries written after it is created, so it is created before
the traffic is sent.

In [ ]:
from google.cloud import monitoring_v3

LOG = f"projects/{PROJECT}/logs/cymbal-agent"
metric = log_client.metric(
    "agent_tool_errors", description="Agent tool calls that failed",
    filter_=f'logName="{LOG}" AND jsonPayload.event="tool_result" '
            'AND jsonPayload.status="error"')
if not metric.exists():
    metric.create()
print("Log-based metric:", metric.name)

Policy = monitoring_v3.AlertPolicy
alerts = monitoring_v3.AlertPolicyServiceClient()
existing = [p.display_name for p in
            alerts.list_alert_policies(name=f"projects/{PROJECT}")]
if "Agent tool errors" not in existing:
    alerts.create_alert_policy(
        name=f"projects/{PROJECT}", alert_policy=Policy(
            display_name="Agent tool errors",
            combiner=Policy.ConditionCombinerType.OR,
            conditions=[Policy.Condition(
                display_name="More than 2 tool errors in 5 minutes",
                condition_threshold=Policy.Condition.MetricThreshold(
                    filter='metric.type="logging.googleapis.com/user/'
                           'agent_tool_errors" AND resource.type="global"',
                    aggregations=[monitoring_v3.Aggregation(
                        alignment_period={"seconds": 300},
                        per_series_aligner=monitoring_v3.Aggregation
                        .Aligner.ALIGN_SUM)],
                    comparison=monitoring_v3.ComparisonType.COMPARISON_GT,
                    threshold_value=2, duration={"seconds": 0}))]))
print("Alert policy: Agent tool errors (more than 2 in 5 minutes)")

## Step 4: Send traffic, including requests that fail

Send five questions. The three camping tent questions fail.

In [ ]:
QUESTIONS = ["Do you have Trailhead boots in stock?",
             "Is the rain jacket available?",
             "Do you have the camping tent in stock?",
             "Is the camping tent back in stock yet?",
             "Can I buy a camping tent today?"]
for question in QUESTIONS:
    message = types.Content(role="user", parts=[types.Part(text=question)])
    for event in runner.run(user_id="lab-user", new_message=message,
                            session_id=str(uuid.uuid4())):
        if event.is_final_response() and event.content:
            print(f"Q: {question}\n   A: {event.content.parts[0].text}")
trace.get_tracer_provider().force_flush()  # Send the spans now.

**Expected result:** two stock answers and three apologies.

## Step 5: Find the slowest request and print its trace waterfall

Each request produced a **trace** made of **spans**. The cell finds the slowest request
and prints its spans, indented by parent, with the time each one started and how long it
took.

In [ ]:
finished = spans.get_finished_spans()
by_id = {s.context.span_id: s for s in finished}
roots = [s for s in finished if s.parent is None]
slowest = max(roots, key=lambda s: s.end_time - s.start_time)


def depth(span):
    level = 0
    while span.parent and span.parent.span_id in by_id:
        span, level = by_id[span.parent.span_id], level + 1
    return level


TRACE_ID = slowest.context.trace_id
print(f"{'Span':<46}{'Starts at':>10}{'Duration':>10}")
for s in sorted(finished, key=lambda s: s.start_time):
    if s.context.trace_id == TRACE_ID:
        start = (s.start_time - slowest.start_time) / 1e6
        ms = (s.end_time - s.start_time) / 1e6
        name = "  " * depth(s) + s.name
        print(f"{name[:44]:<46}{start:>7.0f} ms{ms:>7.0f} ms")
print(f"\nTrace ID: {TRACE_ID:032x}")

**What to look for:** `execute_tool check_inventory` takes about 2,000 ms, so the
warehouse call, not the model, is the bottleneck. The first `generate_content` span stays
open until the tool that the model requested finishes, so compare it with the second
`generate_content` span, which measures only the model's reply.

## Step 6: Query the structured logs for tool results

Query the structured logs, newest first.

In [ ]:
query = f'logName="{LOG}" AND jsonPayload.event="tool_result"'
for wait in range(12):  # New log entries take a few seconds to appear.
    entries = list(log_client.list_entries(
        filter_=query, order_by=cloud_logging.DESCENDING, max_results=20))
    if len(entries) >= len(QUESTIONS):
        break
    time.sleep(5)
for entry in entries:
    p = entry.payload
    print(f"{entry.severity:<8}{p['tool']:<17}{p['status']:<7}"
          f"{p['latency_ms']:>6} ms  trace {entry.trace[-32:][:8]}...")
errors = sum(e.payload["status"] == "error" for e in entries)
print(f"\nTool error rate: {errors}/{len(entries)}")

**Expected result:** three `ERROR` entries of about 2,000 ms and two `INFO` entries of about 200 ms, each with a trace ID, and a tool error rate of 3/5.

## Step 7: Explore in the Console

- In **Trace Explorer**, search for the trace ID from Step 5 and compare the waterfall.
- In **Logs Explorer**, query `jsonPayload.event="tool_result"`, expand an `ERROR` entry, and
  open its trace from the entry.
- After a few minutes, open **Monitoring > Alerting** and confirm that the **Agent tool
  errors** policy has opened an incident. In production, add a notification channel, such as
  email or a paging service.

## Check Your Understanding

**1. Why does each log entry include the trace ID?**

<details><summary>Show answer</summary>

So that an operator can go from a failed log entry straight to the trace that shows every step of that request.

</details>

**2. Why are the logs written as JSON instead of plain text?**

<details><summary>Show answer</summary>

Fields such as tool, status, and latency can be filtered, counted by log-based metrics, and analyzed in BigQuery.

</details>

**3. The p95 latency rises, but the model spans are unchanged. Where do you look?**

<details><summary>Show answer</summary>

At the execute_tool spans and the tool latency field in the logs: a slow tool or dependency is the most common cause.

</details>

## Clean Up

Run the next cell to delete the alert policy and the log-based metric. If you did not clean
up after Lab 09-01, also run the cleanup cell in that notebook.

In [ ]:
import os

from google.cloud import logging as cloud_logging
from google.cloud import monitoring_v3

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
alerts = monitoring_v3.AlertPolicyServiceClient()
for policy in alerts.list_alert_policies(name=f"projects/{PROJECT}"):
    if policy.display_name == "Agent tool errors":
        alerts.delete_alert_policy(name=policy.name)
        print("Alert policy deleted")
metric = cloud_logging.Client(project=PROJECT).metric("agent_tool_errors")
if metric.exists():
    metric.delete()
    print("Log-based metric deleted")
print("Cleanup complete.")

## Summary

You exported traces to Cloud Trace, wrote structured logs linked to traces, created a
log-based metric and an alert policy, and used them to find a slow, failing dependency.